# Meliora: every public method

29 self-contained examples with formulas, assumptions, executable checks and interpretation. Install first (`python -m pip install -e ".[dev]"` from the repository). Each code cell defines its own data. Tiny samples illustrate arithmetic, not reliable asymptotic inference. No external datasets or network access are required.

These examples describe the corrected 0.2 development API. Read the migration guide before comparing with 0.1.2.

## binomial_test

Test for underestimated default probability in each grade.

p_value = P[Binomial(N, mean predicted PD) >= D]. The alternative is underestimated default probability; reject when p_value < alpha_level. Requires independent obligors with a common PD per grade. Averaging heterogeneous PDs is an approximation, not a Poisson-binomial test. Grade-wise decisions are unadjusted for multiplicity.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [1]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.binomial_test(data, 'grade', 'outcome', 'pd')
assert np.allclose(result.p_value, [.1808, .8208])
assert result['Reject H0'].tolist() == [False, False]
result

,Rating class,Predicted PD,Total count,Defaults,Actual Default Rate,p_value,Reject H0
0,A,0.2,4,2.0,0.5,0.1808,False
1,B,0.6,4,2.0,0.5,0.8208,False


**Interpretation.** Neither grade rejects at 5%. Four observations per grade give little power; non-rejection does not establish calibration.

**Invalid or undefined inputs.** Invalid columns, binary outcomes, probabilities, or significance level.

## brier_score

Calculate the observation-level mean squared probability error.

Brier = mean((outcome - predicted PD)**2) over individual observations. Grades are validated but do not alter weighting; ratings is retained for compatibility. This proper scoring rule measures calibration and discrimination, depends on prevalence, and has no null hypothesis or p-value.

[Statistical reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.brier_score_loss.html).

In [2]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.brier_score(data, 'grade', 'outcome', 'pd')
assert np.isclose(result, .30)
result

0.30000000000000004

**Interpretation.** Mean squared probability error is 0.30. Compare models on the same observations and event definition.

**Invalid or undefined inputs.** Invalid columns, nonbinary outcomes, or PDs outside [0, 1].

## herfindahl_test

Measure concentration of one portfolio across rating grades.

For K grades and shares s, CV=sqrt(K*sum((s-1/K)**2)); HHI=sum(s**2), ranging from 1/K to 1. Larger means more concentration. This classic HHI differs from the logarithmic ECB transformation. No p-value or alpha parameter applies. Empty grades in rating_order affect K and CV.

[Statistical reference](https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf).

In [3]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 2})
result = m.herfindahl_test(data, 'grade')
assert np.allclose(result, [1 / 3, 5 / 9])
result

(0.3333333333333333, 0.5555555555555555)

**Interpretation.** HHI=5/9 exceeds the two-grade uniform baseline of 1/2. Compare on the same grade universe.

**Invalid or undefined inputs.** Missing grades or invalid/incomplete rating order.

## herfindahl_multiple_period_test

Test whether grade concentration increased using the ECB CV statistic.

Use the union of grades, retaining zero counts, or rating_order. For initial/current CVs c1/c2, z=sqrt(K-1)*(c2-c1)/sqrt(c2**2*(0.5+c2**2)); p_value=normal.sf(z). Reject increased concentration when p_value < alpha_level. This is the ECB asymptotic CV comparison; HHI columns use classic squared shares. It is undefined for zero current CV and is not a paired-account test.

[Statistical reference](https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf).

In [4]:
import numpy as np
import pandas as pd

import meliora as m

initial = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 2})
current = pd.DataFrame({'grade': ['A'] * 5 + ['B']})
result = m.herfindahl_multiple_period_test(initial, current, 'grade')
assert np.isclose(result.loc['total', 'z_stat'], 3 / np.sqrt(34))
assert np.isclose(result.loc['total', 'h_current'], 26 / 36)
result

,N_initial,N_current,h_initial,h_current,z_stat,p_value,reject
grade,,,,,,,
A,4,5,NaN,NaN,NaN,NaN,<NA>
B,2,1,NaN,NaN,NaN,NaN,<NA>
total,6,6,0.555556,0.722222,0.514496,0.303453,False


**Interpretation.** Concentration increased, but this statistic does not reject at 5%. Six accounts illustrate arithmetic only.

**Invalid or undefined inputs.** Invalid data/order/alpha, fewer than two grades, uniform current shares, or reserved grade label 'total'.

## hosmer_test

Apply a grouped chi-square calibration test to grade-level default counts.

Q=sum((D-N*p)**2/(N*p*(1-p))); p_value=chi2.sf(Q,K-ddof). Default ddof=0 tests fixed externally specified grade PDs with K degrees of freedom. Explicit ddof=2 selects the conventional fitted-logistic Hosmer-Lemeshow approximation with appropriate grouping. Requires independent outcomes, homogeneous grade PDs and sufficiently large expected default/nondefault counts. Reject any departure when p_value < alpha_level.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [5]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.hosmer_test(data, 'grade', 'outcome', 'pd')
assert np.isclose(result[0], np.exp(-(2.25 + 1 / 6) / 2))
assert result[1] is False
result

[0.29869468928867837, False]

**Interpretation.** The fixed-PD test does not reject. Tiny counts make the chi-square inference unreliable.

**Invalid or undefined inputs.** Invalid data/alpha, boundary grade mean PDs, or noninteger ddof outside 0 <= ddof < K.

## spiegelhalter_test

Test probability calibration with the observation-level Spiegelhalter z statistic.

z=sum((y-p)*(1-2*p))/sqrt(sum(p*(1-p)*(1-2*p)**2)), using individual observations. Under independent Bernoulli outcomes with correct fixed PDs the statistic is approximately normal. Reject when 2*normal.sf(abs(z)) < alpha_level. Calibration errors can cancel. PDs of 0, 0.5 or 1 contribute zero null variance; entirely zero variance makes inference undefined.

[Statistical reference](https://doi.org/10.1002/sim.4780050506).

In [6]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4 + ['B'] * 4, 'outcome': [0, 0, 1, 1] * 2, 'pd': [.2] * 4 + [.6] * 4})
result = m.spiegelhalter_test(data, 'grade', 'outcome', 'pd')
assert np.isclose(result[0], .8 / np.sqrt(.2688))
assert result[1] is False
result

(1.5430334996209187, False)

**Interpretation.** The statistic is about 1.54 and does not reject at 5% under the two-sided convention.

**Invalid or undefined inputs.** Invalid data/alpha or zero null variance.

## jeffreys_test

Calculate a Jeffreys-posterior lower-tail probability for each rating grade.

A Beta(1/2,1/2) prior and D defaults in N independent trials yield Beta(D+1/2,N-D+1/2). Return its CDF at mean predicted PD. A small tail indicates predicted PD lies below most posterior mass; Reject H0 uses p_value < alpha_level. This posterior tail is not an exact frequentist p-value. Common grade PDs and independence are assumed; no multiplicity adjustment is applied.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [7]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'grade': ['A'] * 4, 'outcome': [0, 0, 1, 1], 'pd': [.5] * 4})
result = m.jeffreys_test(data, 'grade', 'outcome', 'pd')
assert np.isclose(result.p_value.iloc[0], .5)
result

,Rating class,Predicted PD,Total count,Defaults,Actual Default Rate,p_value,Reject H0
0,A,0.5,4,2.0,0.5,0.5,False


**Interpretation.** The symmetric posterior has lower-tail probability 0.5 at the forecast PD.

**Invalid or undefined inputs.** Invalid columns, outcomes, probabilities, or significance level.

## roc_auc

Calculate binary ROC area, assigning half credit to tied scores.

AUC=P(score of outcome 1 > score of outcome 0)+0.5*P(tie). Larger finite scores must mean greater default risk; scores need not be probabilities. Both binary classes are required. Uses scikit-learn ROC AUC. This descriptive measure has no p-value and does not measure calibration.

[Statistical reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

In [8]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.roc_auc(data, 'y', 'score')
assert np.isclose(result, .875)
result

0.875

**Interpretation.** Three positive-negative pairs are correctly ordered and one ties: (3+0.5)/4=0.875.

**Invalid or undefined inputs.** Invalid columns, nonfinite scores, nonbinary target or an absent class.

## gini

Calculate the normalized discrimination Gini as twice ROC AUC minus one.

Gini=2*ROC_AUC-1, with higher scores for outcome 1 and half credit for ties. Values 1, 0 and -1 indicate perfect, random and reverse ordering. Requires both classes and finite scores. This is the discrimination Gini, not an income-inequality estimator or a calibration significance test.

[Statistical reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_auc_score.html).

In [9]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.gini(data, 'y', 'score')
assert np.isclose(result, .75)
result

0.75

**Interpretation.** An AUC of 0.875 corresponds to Gini 0.75.

**Invalid or undefined inputs.** Invalid classification data; see roc_auc.

## kolmogorov_smirnov_stat

Compare score distributions of the two outcome classes using two-sample KS.

D is the largest absolute difference between score CDFs conditional on outcome 0 and outcome 1. Null: equal score distributions; alternative: two-sided. SciPy ks_2samp selects exact/asymptotic inference automatically. It assumes independent samples; continuous-distribution p-values are approximate for tied/discrete scores. D measures separation irrespective of score direction, not PD calibration.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html).

In [10]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 3, 4]})
result = m.kolmogorov_smirnov_stat(data, 'y', 'score')
assert np.isclose(result.statistic, 1)
assert np.isclose(result.pvalue, 1 / 3)
result

KstestResult(statistic=np.float64(1.0), pvalue=np.float64(0.3333333333333333), statistic_location=np.float64(2.0), statistic_sign=np.int8(1))

**Interpretation.** Class score ranges do not overlap (D=1), but the exact two-sided p-value is 1/3 for two observations per class.

**Invalid or undefined inputs.** Invalid classification columns, scores or classes.

## cumulative_lgd_accuracy_ratio

Calculate the VUROCS cumulative LGD accuracy measure for ordinal grades.

At each threshold from highest to lowest grade, x=P(predicted >= threshold), y=P(predicted >= threshold AND realised >= threshold). Include (0,0), integrate whole tied-grade bands by trapezoids and return twice the area. This follows VUROCS clar (Ozdemir and Miu convention). It is ordinal accuracy, not a chance-adjusted Gini. Predictions and outcomes share a grade scale, higher meaning more loss. Accounts have equal weight; no p-value applies.

[Statistical reference](https://cran.r-universe.dev/VUROCS/VUROCS.pdf).

In [11]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'p': [1, 2, 3, 3, 4], 'y': [1, 3, 2, 4, 4]})
result = m.cumulative_lgd_accuracy_ratio(data, 'p', 'y')
assert np.isclose(result, .88)
result

0.8800000000000001

**Interpretation.** Threshold points (0,0), (.2,.2), (.6,.4), (.8,.8), (1,1) give twice the trapezoidal area 0.88.

**Invalid or undefined inputs.** Invalid columns, missing grades or invalid/ambiguous grade order.

## loss_capture_ratio

Compare model and ideal loss-capture gains using cumulative exposure on the x axis.

Sort predicted LGD descending; x=cumulative EAD share, y=cumulative realised monetary loss share. Pool tied scores and include the origin. The ideal curve sorts realised LGD; LCR=(area_model-0.5)/(area_ideal-0.5). This explicitly uses exposure on the x axis; account-count variants differ. Perfect/reverse ordering yields 1/-1; constant predicted scores yield 0. Zero EAD has no influence. This is descriptive with no p-value.

[Statistical reference](https://aptivaa.com/pdf/ifrs9-model-risk-management-1594098423-1.pdf).

In [12]:
import numpy as np
import pandas as pd

import meliora as m

result = m.loss_capture_ratio([1, 1, 1], [.1, .4, .9], [.1, .4, .9])
assert np.isclose(result, 1)
result

1.0

**Interpretation.** Sorting on true loss rates reproduces the ideal curve. Match the area convention when comparing implementations.

**Invalid or undefined inputs.** Invalid paired LGDs/exposures, nonpositive total EAD/loss, or zero ideal gain (constant LGD on positive exposures).

## bayesian_error_rate

Find the minimum empirical misclassification rate over score thresholds.

For every threshold, error=(1-prevalence)*FPR+prevalence*(1-TPR). Return the minimum, including all/none-positive predictions. This is empirical threshold-optimized error with equal costs, not irreducible Bayes error. Higher scores mean outcome 1. Training-set optimization is optimistic: evaluate on held-out data. Requires both classes; returns no p-value.

[Statistical reference](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html).

In [13]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'y': [0, 0, 1, 1], 'score': [1, 2, 2, 3]})
result = m.bayesian_error_rate(data, 'y', 'score')
assert np.isclose(result, .25)
result

0.25

**Interpretation.** A positive and negative tie at score 2, so at least one of four observations is misclassified.

**Invalid or undefined inputs.** Invalid classification data or an absent outcome class.

## information_value

Measure separation of pre-binned feature distributions between binary outcomes.

Pre-bin the feature. Add smoothing to every bin/class count and normalize classes separately. WoE=log(good_share/bad_share); IV=sum((good_share-bad_share)*WoE). Outcome 0 is good, 1 bad. Default smoothing=0.5 makes empty cells finite; smoothing=0 requires positive cells. Uses the union of observed bins. Descriptive and nonnegative, with no universal acceptance threshold or p-value. Sparse or selected bins can inflate IV.

[Statistical reference](https://doi.org/10.1214/aoms/1177729694).

In [14]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'bin': ['A'] * 4 + ['B'] * 4, 'y': [0, 0, 0, 1, 0, 1, 1, 1]})
result = m.information_value(data, 'bin', 'y', smoothing=0)
assert np.isclose(result[1], np.log(3))
assert np.allclose(result[0][["good_share", "bad_share"]].sum(), 1)
result

(     good  bad  good_share  bad_share       WoE        IV
 bin                                                      
 A     3.0  1.0        0.75       0.25  1.098612  0.549306
 B     1.0  3.0        0.25       0.75 -1.098612  0.549306,
 1.0986122886681098)

**Interpretation.** All cells are positive, allowing smoothing=0. The unsmoothed IV is log(3).

**Invalid or undefined inputs.** Invalid columns/classes/smoothing, or zero cells with smoothing=0.

## lgd_t_test

Test whether mean realised LGD exceeds mean expected LGD using paired errors.

Paired errors e=realised-expected LGD; t=mean(e)/sqrt(sample_variance(e)/N); p_value=t.sf(t,N-1). The one-sided alternative is underestimation (positive mean error). Independent normal errors give exact finite-sample t inference. Each segment needs at least two observations and nonzero error variance. Segment p-values are unadjusted. Each account has equal weight.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html).

In [15]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.lgd_t_test(data, 'realised', 'predicted', level='segment', segment_col='segment')
assert result.segment.tolist() == ['A', 'B']
assert np.isclose(result.loc[0, 'p_value'], .5)
assert result.loc[1, 't_stat'] < 0
result

,segment,N,realised_lgd_mean,pred_lgd_mean,s2,mean_error,t_stat,p_value
0,A,2,0.30,0.3,0.020,-1.387779e-17,-1.387779e-16,0.500000
1,B,2,0.65,0.7,0.045,-5.000000e-02,-3.333333e-01,0.602416


**Interpretation.** Segment A has zero average error and p=0.5. Segment B has negative mean error, opposite the underestimation alternative.

**Invalid or undefined inputs.** Invalid level/segment, columns/LGDs, fewer than two observations per group or zero error variance.

## migration_matrix_stability

Calculate ECB adjacent-cell migration z statistics and their normal CDFs.

For off-diagonal probability f and adjacent probability n one step nearer the diagonal, z=(n-f)/sqrt((f*(1-f)+n*(1-n)+2*f*n)/N_i). Return Phi(z), as in ECB 2019 instructions. Small CDFs indicate violations of decreasing off-diagonal mass. These are asymptotic multinomial comparisons, not time-series equality tests. Retain empty grades. NaN means undefined, not passed. Cell probabilities are not multiplicity-adjusted.

[Statistical reference](https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf).

In [16]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'start': [1] * 4 + [2] * 4 + [3] * 4, 'end': [1, 1, 2, 3, 1, 2, 2, 3, 1, 2, 3, 3]})
result = m.migration_matrix_stability(data, 'start', 'end')
assert np.isclose(result[0].loc[1, 2], 2 / np.sqrt(11))
assert np.isnan(np.diag(result[0])).all()
result

(end           1         2         3
 start                              
 1           NaN  0.603023  0.000000
 2      0.603023       NaN  0.603023
 3      0.000000  0.603023       NaN,
 end           1         2         3
 start                              
 1           NaN  0.726753  0.500000
 2      0.726753       NaN  0.726753
 3      0.500000  0.726753       NaN)

**Interpretation.** Cell (1,2) compares 2/4 on the diagonal against 1/4 nearby. Cell (1,3) has equal adjacent probabilities, giving CDF 0.5.

**Invalid or undefined inputs.** Invalid grade columns or rating order.

## population_stability_index

Compare two distributions on a common set of pre-defined bins using PSI.

Require exactly two samples and shared pre-defined bins. Add smoothing per sample/bin cell, normalize samples to shares E,A, then PSI=sum((A-E)*log(A/E)). Defaults choose expected/actual in natural or categorical order; explicit labels are clearer. Default smoothing=0.5; zero requires positive cells. PSI is symmetric and descriptive, not a significance test. Binning and smoothing change the value; no universal cutoffs are imposed.

[Statistical reference](https://doi.org/10.1214/aoms/1177729694).

In [17]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'period': ['old'] * 4 + ['new'] * 4, 'bin': ['A', 'A', 'A', 'B', 'A', 'B', 'B', 'B']})
result = m.population_stability_index(data, 'period', 'bin', expected='old', actual='new', smoothing=0)
assert np.isclose(result[1], np.log(3))
assert np.allclose(result[0][["expected", "actual"]].sum(), 1)
result

(     expected  actual       PSI
 bin                            
 A        0.75    0.25  0.549306
 B        0.25    0.75  0.549306,
 1.0986122886681098)

**Interpretation.** Shares change from (.75,.25) to (.25,.75), giving PSI=log(3). Significance requires a separate sampling model.

**Invalid or undefined inputs.** Missing columns, other than two samples, invalid/partial sample labels, invalid smoothing or zero unsmoothed cells.

## kendall_tau

Calculate Kendall ordinal association with the requested tie normalization.

Kendall tau compares concordant and discordant pairs. Variant b adjusts ties in both variables; c normalizes using distinct categories. Both equal tau-a without ties. The coefficient lies in [-1,1]. Independent pairs are needed for inference. SciPy selects exact/asymptotic inference as applicable; missing values are never silently omitted.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kendalltau.html).

In [18]:
import numpy as np
import pandas as pd

import meliora as m

result = m.kendall_tau([1, 2, 3, 4], [1, 2, 4, 8])
assert np.isclose(result[0], 1)
assert np.isclose(result[1], 1 / 12)
result

(1.0, 0.08333333333333333)

**Interpretation.** All pairs concord: tau=1. The exact two-sided p-value for four distinct observations is 2/4!=1/12.

**Invalid or undefined inputs.** Invalid/nonconstant paired vectors or unsupported variant.

## somersd

Calculate asymmetric Somers D of the second variable conditional on the first.

Somers D divides concordance minus discordance by pairs untied in the first (row/independent) variable. Swapping inputs can change D. Accept ranking vectors or a contingency table with ordered rows/columns. The null is D=0, with two-sided/less/greater alternatives. SciPy uses asymptotic normal inference for independent pairs; sparse tables weaken the approximation.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.somersd.html).

In [19]:
import numpy as np
import pandas as pd

import meliora as m

result = m.somersd([[3, 1], [1, 3]])
assert np.isclose(result.statistic, .5)
result

SomersDResult(statistic=np.float64(0.5), pvalue=np.float64(0.10247043485974937), table=array([[3., 1.],
       [1., 3.]]))

**Interpretation.** Row-conditional association is 0.5. Tests also use asymmetric tables to check directionality.

**Invalid or undefined inputs.** Invalid vectors/alternative/table; tables need nonnegative integer counts and at least two nonempty rows and columns.

## spearman_correlation

Calculate Spearman rank correlation and an asymptotic association p-value.

Spearman correlation is Pearson correlation of average ranks. It measures monotone association and handles ties using average ranks. SciPy returns an asymptotic p-value for zero rank correlation under the chosen alternative. Small-sample p-values are unreliable; consider permutation inference. Require at least three independent pairs and no constant arrays.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html).

In [20]:
import numpy as np
import pandas as pd

import meliora as m

result = m.spearman_correlation([1, 2, 3, 4], [1, 2, 4, 8])
assert np.isclose(result.statistic, 1)
result

SignificanceResult(statistic=np.float64(1.0), pvalue=np.float64(0.0))

**Interpretation.** The relationship is strictly increasing: rank correlation equals 1 despite a nonlinear scale.

**Invalid or undefined inputs.** Invalid alternative/vectors, fewer than three pairs, or constant inputs.

## pearson_correlation

Calculate Pearson product-moment correlation and its association p-value.

Pearson r is the centered cross-product divided by the product of centered Euclidean norms, in [-1,1]. Calculation does not require normality; SciPy default p-values assume independent bivariate-normal pairs under zero correlation. Near-constant inputs can emit SciPy NearConstantInputWarning; inspect and rescale data. Alternatives are two-sided, less or greater.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html).

In [21]:
import numpy as np
import pandas as pd

import meliora as m

result = m.pearson_correlation([1, 2, 3, 4], [1, 2, 4, 8])
assert np.isclose(result.statistic, 11.5 / np.sqrt(143.75))
assert result.statistic < 1
result

PearsonRResult(statistic=np.float64(0.9591663046625439), pvalue=np.float64(0.04083369533745618))

**Interpretation.** Pearson r is about 0.959, below Spearman r=1: monotone does not imply exactly linear.

**Invalid or undefined inputs.** Invalid alternative/vectors, fewer than two pairs, or constant inputs.

## migration_matrices_statistics

Calculate ECB normalized migration-weighted bandwidth above and below the diagonal.

For each side of the diagonal, divide sum(|i-j|*N_ij) by sum(max(i,K-1-i)*N_ij), using zero-based grades. This is ECB normalized migration-weighted bandwidth. Upper means a later grade in rating_order; economic direction depends on that order. A side with no migrations has bandwidth 0 by convention. Describes distance, not frequency or significance.

[Statistical reference](https://www.bankingsupervision.europa.eu/activities/internal_models/shared/pdf/instructions_validation_reporting_credit_risk.en.pdf).

In [22]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'start': [1] * 4 + [2] * 4 + [3] * 4, 'end': [1, 1, 2, 3, 1, 2, 2, 3, 1, 2, 3, 3]})
result = m.migration_matrices_statistics(data, 'start', 'end')
assert np.allclose(result, [.8, .8])
result

(0.8, 0.8)

**Interpretation.** Each side has distance-weighted count 4 and maximum-distance-weighted count 5, giving 0.8.

**Invalid or undefined inputs.** Invalid grade columns or rating order.

## conditional_information_entropy_ratio

Measure the fraction of marginal default uncertainty explained by grades.

For rates p_i and normalized counts w_i, H0=h(sum(w_i*p_i)), H1=sum(w_i*h(p_i)), h(p)=-p*log(p)-(1-p)*log(1-p). Use natural logs and 0*log(0)=0. This is the fraction of binary uncertainty explained by grades. Zero counts contribute nothing; fractional weights are permitted. Descriptive, not a calibration p-value. A deterministic portfolio outcome makes the denominator zero.

[Statistical reference](https://doi.org/10.1002/j.1538-7305.1948.tb01338.x).

In [23]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'rate': [0, 1], 'n': [10, 10]})
result = m.conditional_information_entropy_ratio(data, 'rate', 'n')
assert np.isclose(result, 1)
result

1.0

**Interpretation.** Deterministic grade outcomes explain all marginal uncertainty in this balanced portfolio.

**Invalid or undefined inputs.** Invalid rates/counts/columns, nonpositive total count, or marginal rate 0 or 1.

## kullback_leibler_dist

Calculate mutual information between rating grade and binary default outcome.

The historical name denotes grade/default mutual information, not a general two-distribution KL function. Compute marginal minus count-weighted conditional binary entropy, with natural logs and 0*log(0)=0. Equivalently, average grade Bernoulli KL divergences from the portfolio Bernoulli distribution. Zero weights contribute nothing; all-zero/all-one outcomes return 0. No p-value applies.

[Statistical reference](https://doi.org/10.1214/aoms/1177729694).

In [24]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'rate': [0, 1], 'n': [10, 10]})
result = m.kullback_leibler_dist(data, 'rate', 'n')
assert np.isclose(result, np.log(2))
result

0.6931471805599453

**Interpretation.** Perfect grade separation removes log(2) nats of uncertainty from a balanced binary outcome.

**Invalid or undefined inputs.** Invalid columns/rates/counts or nonpositive total count.

## loss_shortfall

Calculate relative underestimation of total exposure-weighted monetary loss.

Monetary loss is EAD*LGD. Positive means aggregate underestimation, zero equal totals, negative overestimation. Nonnegative exposures need positive total; realised total loss must be positive. This descriptive aggregate can hide offsetting errors and is not a significance test.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [25]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.loss_shortfall(data, 'ead', 'predicted', 'realised')
assert np.isclose(result, 0)
result

0.0

**Interpretation.** Predicted and realised losses both total 240. MAD reveals individual errors hidden by this equality.

**Invalid or undefined inputs.** Invalid columns/LGDs/exposures or nonpositive total EAD/realised loss.

## mean_absolute_deviation

Calculate the exposure-weighted mean absolute LGD prediction error.

MAD=sum(EAD*abs(realised-predicted LGD))/sum(EAD). Zero exposures do not contribute. Both LGDs use [0,1] fractions. This descriptive score measures error magnitude without cancellation; no hypothesis or p-value applies.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [26]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.mean_absolute_deviation(data, 'ead', 'predicted', 'realised')
assert np.isclose(result, .12)
result

0.11999999999999998

**Interpretation.** Absolute monetary error is 60 on EAD 500: MAD=0.12, or 12 LGD percentage points.

**Invalid or undefined inputs.** Invalid columns/LGDs/exposures, negative EAD or nonpositive total EAD.

## elbe_t_test

Test equality of mean realised LGD and ELBE using a two-sided paired t test.

For paired realised LGD minus ELBE errors, t=mean(error)/sqrt(sample_variance(error)/N). Return a two-sided t p-value with N-1 degrees of freedom. Null: zero mean paired error. Independent normal errors support finite-sample inference; each facility has equal weight. Small p-values can indicate either underestimation or overestimation.

[Statistical reference](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ttest_rel.html).

In [27]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'ead': [100, 200, 100, 100], 'predicted': [.2, .4, .6, .8], 'realised': [.1, .5, .4, .9], 'segment': ['A', 'A', 'B', 'B']})
result = m.elbe_t_test(data, 'realised', 'predicted')
assert np.isclose(result.lgd_mean.iloc[0], .475)
assert result.t_stat.iloc[0] < 0
assert 0 < result.p_value.iloc[0] < 1
result

,facilities,lgd_mean,elbe_mean,t_stat,p_value
0,4,0.475,0.5,-0.333333,0.76082


**Interpretation.** Realised mean LGD is 0.475 versus ELBE 0.5. This equal-facility test differs from an exposure-weighted score.

**Invalid or undefined inputs.** Invalid columns/LGDs, fewer than two pairs, or zero error variance.

## normal_test

Apply the Basel one-sided normal approximation to annual PD forecast errors.

For T annual observations of one grade, e=realised-predicted PD; s2=sum((e-mean(e))**2)/(T-1). Basel z=sum(e)/sqrt(T*s2); p_value=normal.sf(z). Alternative: PD underestimation; reject when p_value < alpha. Assumes independent annual errors with common finite variance and sufficient years. Uses a normal, not Student t, reference; preserves historical column t_stat. Not an obligor-level Bernoulli test.

[Statistical reference](https://www.bis.org/publ/bcbs_wp14.pdf).

In [28]:
import numpy as np
import pandas as pd

import meliora as m

result = m.normal_test([.1, .1, .1, .1], [.1, .2, .3, .4])
assert np.isclose(result.t_stat.iloc[0], .6 / np.sqrt(1 / 15))
assert bool(result.outcome.iloc[0])
result

,estimate,t_stat,p_value,outcome
0,0.15,2.32379,0.010068,True


**Interpretation.** Errors 0, 0.1, 0.2, 0.3 give z about 2.324 and p about 0.010. Four years illustrate arithmetic, not a strong asymptotic basis.

**Invalid or undefined inputs.** Invalid alpha/rates/pairing, fewer than two years or zero annual-error variance.

## redelmeier_test

Compare paired Brier losses under an explicit midpoint-Bernoulli null model.

This Redelmeier-style comparison explicitly assumes independent Y_i~Bernoulli(q_i), q_i=(p1_i+p2_i)/2, with fixed forecasts. Paired squared-loss difference is (p1-p2)*(p1+p2-2Y), with null mean zero and variance (p1-p2)**2*(p1+p2)*(2-p1-p2). Sum differences and variances to form z; return 2*normal.sf(abs(z)). Positive z favors the second forecast. This normal approximation uses the stronger midpoint null, not unrestricted equality of average Brier scores. The variance is derived from this stated model; the original paper motivates paired Brier comparisons, not certification of this convention. Identical forecasts provide no comparative evidence.

[Statistical reference](https://pubmed.ncbi.nlm.nih.gov/1941009/).

In [29]:
import numpy as np
import pandas as pd

import meliora as m

data = pd.DataFrame({'y': [0, 1, 1, 0], 'p1': [.1, .4, .7, .3], 'p2': [.2, .6, .6, .1]})
result = m.redelmeier_test(data, default_flag='y', first_pd='p1', second_pd='p2')
assert np.isclose(result[0], .18 / np.sqrt(.0798))
assert 0 < result[1] < 1
result

(0.6371930928643097, 0.523999076234716)

**Interpretation.** First-forecast total squared loss exceeds the second by 0.18. Swapping forecasts reverses z and preserves the two-sided p-value.

**Invalid or undefined inputs.** Missing columns, nonbinary outcomes or invalid probabilities.